# Nuc2D: Visualize RNA and DNA secondary structures

Nuc2D (`nuc2d`) は，**RNA・DNA の二次構造を描画** するための **Python ライブラリ** です。

<p align="center">
  <img src="https://raw.githubusercontent.com/Soma-yu/nuc2d/main/docs/images/example.png" width="75%">
</p>

<a id="toc"></a>

## 目次

| | | |
|---|---|---|
| [0. はじめに](#sec0) | このファイルの使い方 | |
| [1. 二次構造を描く](#sec1) | 文字列から二次構造を描画 | `draw_svg` |
| [2. 塩基配列を載せる](#sec2) | ノードの中に塩基を表示 | `sequences` |
| [3. 塩基対形成確率を示す](#sec3) | ノードの色で確率を表現 | `probs` |
| [4. 複数の二次構造を並べる](#sec4) | より自由で高度な描画 | `draw_component` / `Placement` / `compose` |
| [5. 保存と見た目の調整](#sec5) | 保存・色・間隔 | `DrawingStyle` / `RadialLayoutEngine` |


---

<a id="sec0"></a>

## 0. はじめに

このファイルでは **実際にコードを実行して** 二次構造の描画を体験できます。
Google Colab 上で動くので，**手元の PC には何も入れる必要がありません**。

### 使い方

**上から順に各セルの左上の実行ボタンを押すだけ**です。
(もしくは上部メニューの「ランタイム」から「**すべてのセルを実行**」を選ぶ。)

各章は前の章の変数を使うので，飛ばさずに順に実行してください。
一度通したあとは，構造や配列の文字列を自分のものに書き換えて試せます。

In [ ]:
# 「#」から行末まではコメント。Pythonは読み飛ばすので，消しても動きは変わらない

# nuc2dを取ってくる (Colabは開くたびに初期化されるので，毎回必要)
!pip install -q nuc2d

import matplotlib as mpl
import numpy as np
import svgwrite
from IPython.display import SVG, display

import nuc2d
from nuc2d.font import find_font_path, vertical_center_offset

<a id="sec1"></a>

## 1. 二次構造を描く

核酸の二次構造は **dot-bracket 記法** で書かれた **文字列** で表現できます。

| 文字 | 意味 |
|---|---|
| `(` `)` | 対を組んだ 2 つの塩基 |
| `.` | 対を組んでいない塩基 |
| `+` | 鎖の切れ目 |

Nuc2D (`nuc2d`) を使えば，**dot-bracket 文字列** が表現する二次構造を簡単に描画できます。

In [ ]:
# 2本の鎖(38+38 nt)に分かれたtRNAのクローバーリーフ構造を表現するdot-bracket文字列
CLOVERLEAF = "(((((((..((((........)))).(((((.......+))))).....(((((.......))))))))))))...."

# nuc2dのdraw_svg()という関数がdot-bracket文字列をSVGに変換する
drawing = nuc2d.draw_svg(dot_bracket=CLOVERLEAF)

# SVGの表示
display(SVG(drawing.tostring()))

矢印が付いているのが **3' 末端**です。

二次構造として成り立たない dot-bracket 文字列を渡すと， `ParseError` と呼ばれる**エラー**になります。

In [ ]:
try:
    nuc2d.draw_svg("(((")
except nuc2d.ParseError as error:
    print(error)

<a id="sec2"></a>

## 2. 塩基配列を載せる

`sequences` に **各鎖の塩基配列のリスト** を渡すと，各ノードの中に塩基の種類が描かれます。

In [ ]:
# 各鎖の塩基配列
SEQ1 = "GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGA"
SEQ2 = "UCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA"

# 二次構造と塩基配列をまとめて1つのSVGに変換する
# CLOVERLEAF: dot-bracket文字列 (第1章)
drawing = nuc2d.draw_svg(dot_bracket=CLOVERLEAF, sequences=[SEQ1, SEQ2])

# SVGの表示
display(SVG(drawing.tostring()))

塩基配列の本数や長さが dot-bracket 文字列と合っていなければ `ValueError` と呼ばれる **エラー** になります。

<a id="sec3"></a>

## 3. 塩基対形成確率を示す

`probs` に **塩基対形成確率の行列** を渡すと，各ノードが確率に応じて色分けされ，
横にカラーバーが付きます。

- `probs[i][j]` … 塩基 `i` と `j` が対を組む確率
- `probs[i][i]` … 塩基 `i` が対を組まない確率

本来はこの行列を構造予測ツールが出してくれますが，ここでは説明のために作ります。

In [ ]:
# 説明用の確率行列を生成する関数
def demo_probs(dot_bracket, confidence):
    """説明用の確率行列を作る。ステムごとに確率を変える。

    confidence は (開始位置, 終了位置, 確率) の並び。
    残りは「対を組まない確率」として対角成分に置く。
    """
    flat = dot_bracket.replace("+", "")
    probs = np.zeros((len(flat), len(flat)))

    stack, pairs = [], []
    for i, char in enumerate(flat):
        if char == "(":
            stack.append(i)
        elif char == ")":
            pairs.append((stack.pop(), i))

    for left, right in pairs:
        p = next(v for lo, hi, v in confidence if lo <= left <= hi)
        probs[left][right] = probs[right][left] = p

    for i in range(len(flat)):
        probs[i][i] = max(0.0, 1.0 - probs[i].sum())

    return probs


# 説明用の確率行列(本来ならば構造予測ツールで計算)
probs = demo_probs(CLOVERLEAF, [
    (0, 6, 0.97),    # アクセプターステム: ほぼ確実
    (9, 12, 0.55),   # D アーム: 半々
    (26, 30, 0.88),  # アンチコドンアーム: 高い
    (48, 52, 0.38),  # T アーム: 低い
])

# 二次構造と塩基配列と塩基対形成確率をまとめて1つのSVGに変換する
# CLOVERLEAF: dot-bracket文字列 (第1章)
# [SEQ1, SEQ2]: 各鎖の塩基配列のリスト (第2章)
drawing = nuc2d.draw_svg(dot_bracket=CLOVERLEAF, sequences=[SEQ1, SEQ2], probs=probs)

# SVGの表示
display(SVG(drawing.tostring()))

どのステムがどれくらい安定かが **一目で** 分かります。

カラーバーのラベルは引数 `colorbar_label` で変えられます。

<a id="sec4"></a>

## 4. 複数の二次構造を並べる

二次構造を **部品 (component)** として描くことで，**より高度な描画** が可能です。

使うのは以下の 3 つで，いずれも `nuc2d` に入っています。

- `draw_component` … 構造を 1 つの部品として描く
- `Placement` … どの部品をどこに，どれだけ拡大して置くか
- `compose` … 置いた部品をまとめて，1 つの大きな部品にする

部品はすべて，`svgwrite` で作った **1 つの SVG** に描き込みます。

In [ ]:
# ---- 並べ方の設定 ----
PANEL_HEIGHT = 370.0  # 各部品をこの高さにそろえる
GAP = 20.0            # 部品どうしの間隔


def title_component(
    drawing, component, title, font_family="Arial", font_size=15.0, title_height=25.0
):
    """部品の上に見出しを載せて，ひとつの部品にして返す。

    見出しが部品の幅より長いと，左右にはみ出す。
    """
    box = component.bbox

    # 文字の上下中央はベースラインより上にあるので，その分だけベースラインを下げる。
    # ずらす量はフォントとその大きさから決まり，nuc2dが計算してくれる
    baseline_offset = vertical_center_offset(
        find_font_path(font_family), font_size
    )

    # 見出しの文字
    group = drawing.g()
    group.add(
        drawing.text(
            title,
            insert=(
                (box.xmin + box.xmax) / 2,
                box.ymin - title_height / 2 + baseline_offset,
            ),
            text_anchor="middle",
            font_family=font_family,
            font_size=font_size,
            fill="black",
        )
    )

    # 文字と，それが占める範囲を，ひとつの部品にする
    title_band = nuc2d.SVGComponent(
        group=group,
        bbox=nuc2d.BBox(box.xmin, box.ymin - title_height, box.xmax, box.ymin),
    )

    # 構造の部品と見出しの部品を重ねて，ひとつの部品にする
    return nuc2d.compose(
        drawing.g(),
        [nuc2d.Placement(component=component), nuc2d.Placement(component=title_band)],
    )


# すべての部品を描き込む先のSVG
drawing = svgwrite.Drawing()

# 第1〜3章の描画内容を，見出しを付けた部品にする
panels = [
    title_component(
        drawing,
        nuc2d.draw_component(drawing, dot_bracket=CLOVERLEAF),
        "Secondary structure",
    ),
    title_component(
        drawing,
        nuc2d.draw_component(drawing, dot_bracket=CLOVERLEAF, sequences=[SEQ1, SEQ2]),
        "+ Sequences",
    ),
    title_component(
        drawing,
        nuc2d.draw_component(
            drawing, dot_bracket=CLOVERLEAF, sequences=[SEQ1, SEQ2], probs=probs
        ),
        "+ Sequences, Probabilities",
    ),
]

# 高さをそろえて，左から順に間隔をあけて置く
placements, cursor_x = [], 0.0
for panel in panels:
    scale = PANEL_HEIGHT / panel.bbox.height

    # Placementのx, yは，拡大後の部品をどれだけ平行移動するかを意味する
    # 拡大後の左上を (cursor_x, 0) に合わせたいので，
    # 拡大後の左上 (bbox.xmin * scale, bbox.ymin * scale) を引く
    placements.append(
        nuc2d.Placement(
            component=panel,
            x=cursor_x - panel.bbox.xmin * scale,
            y=0.0 - panel.bbox.ymin * scale,
            scale=scale,
        )
    )

    # 次の部品は，いま置いた部品の拡大後の幅+ギャップ幅だけ右へ
    cursor_x += panel.bbox.width * scale + GAP

# 置いた部品をまとめて，1つの大きな部品にする
row = nuc2d.compose(drawing.g(), placements)

# 大きな部品をSVGに載せ，表示する大きさを部品の範囲に合わせる
drawing.add(row.group)
drawing.viewbox(*row.bbox.to_viewbox())
drawing["width"] = f"{row.bbox.width}px"
drawing["height"] = f"{row.bbox.height}px"

# 全てをまとめて1つにしたSVGを表示
display(SVG(drawing.tostring()))

> **図の中の文字について**
> 図中の文字は `font_family` で指定したフォント (既定は Arial) で描かれます。
> 日本語を入れると環境によっては表示されないので，
> 見出しは英数字にして，説明は図の外に書くのが安全です。

<a id="sec5"></a>

## 5. 保存と見た目の調整

`saveas` で .svg ファイルとして保存できます。
`draw_svg` が返したものでも，第 4 章のように組み立てたものでも同じです。

> Google Colab での保存先は一時的な作業領域です。
> 左側のフォルダのアイコンを開くと保存したファイルが見えるので，
> 手元のPCに残したいときはそこからダウンロードしてください。
> ダウンロードしていないファイルは，セッションが切れると自動的に消去されます。

In [ ]:
# 見出しを付けたSVGを作って保存する

drawing = svgwrite.Drawing()

# 第3章と同じ図に，第4章のtitle_component()で見出しを付ける
panel = title_component(
    drawing,
    nuc2d.draw_component(
        drawing, dot_bracket=CLOVERLEAF, sequences=[SEQ1, SEQ2], probs=probs
    ),
    "tRNA cloverleaf",
)

# 部品をSVGに載せ，表示する大きさを部品の範囲に合わせる
drawing.add(panel.group)
drawing.viewbox(*panel.bbox.to_viewbox())
drawing["width"] = f"{panel.bbox.width}px"
drawing["height"] = f"{panel.bbox.height}px"

# ファイルに保存する
drawing.saveas("cloverleaf.svg")
print("cloverleaf.svg を保存しました")

# SVGの表示
display(SVG(drawing.tostring()))

色・線の太さ・ノードの大きさは `DrawingStyle` で，
塩基どうしの間隔は `RadialLayoutEngine` で変えられます。

`probs` を渡したときのノードの色は，`DrawingStyle` の `cmap` に渡した
**カラーマップ** が決めます。既定は `turbo` で，下の例では `viridis` にしています。
(`probs` を渡さないときは，ノードは `node_color` の一色になります。)

In [ ]:
drawing = nuc2d.draw_svg(
    dot_bracket=CLOVERLEAF,
    sequences=[SEQ1, SEQ2],
    probs=probs,
    style=nuc2d.DrawingStyle(
        backbone_color="#333333",       # 骨格の色
        basepair_color="crimson",       # 塩基対の色
        node_radius=5.0,                # ノードの大きさ
        cmap=mpl.colormaps["viridis"],  # probsの配色
    ),
    layout_engine=nuc2d.RadialLayoutEngine(
        backbone_spacing=18.0,          # 骨格に沿った塩基の間隔
        loop_spacing=24.0,              # ループ上の塩基の間隔
    ),
    width_px=500,   # SVGの横幅
    height_px=500,  # SVGの縦幅。片方だけ指定すると，もう片方は縦横比から決まる
)

# SVGの表示
display(SVG(drawing.tostring()))


---

## もっと詳しく

- GitHubリポジトリ: https://github.com/Soma-yu/nuc2d
- PyPI: https://pypi.org/project/nuc2d/

[目次に戻る](#toc)